# Atari RL Playground：强化学习与持续学习教程

本 Notebook 是仓库的**薄教学入口**：它负责文字、公式、课堂顺序和结果展示；所有可执行功能均来自经过测试的 Python 模块与脚本。

- 算法实现：`algorithms/`
- 环境实现：`environments/`
- 短教学示例：`scripts/tutorial_examples.py`
- 正式训练与评估：`scripts/train_*.py`、`scripts/evaluate.py`
- 批量实验：`scripts/run_experiments.py`

Notebook 不安装依赖、不复制训练循环，也不在课堂单元格中启动长时间训练。

## 1. 环境准备

普通环境按照 README 安装：

```bash
python -m pip install -e ".[dev]"
```

请在启动 Notebook 之前完成安装。当前开发基线为 PyTorch 2.14、TorchRL 0.13.3 和 TensorDict 0.13；PyTorch wheel 自带 CUDA runtime，但运行机器仍需提供兼容的 NVIDIA 驱动。

## 2. 运行环境

下面只调用 Python 模块读取版本和设备信息。受限沙箱中出现 `Can't initialize NVML` 不等于 CUDA 计算不可用。

In [ ]:
from scripts.tutorial_examples import (
    run_dqn_update_demo, run_ewc_penalty_demo, run_gae_demo, runtime_summary,
)

runtime_summary()

## 3. DQN：从 transition 到 Bellman update

对于 transition $(s_t, a_t, r_t, s_{t+1}, d_t)$，本仓库使用目标网络构造

$$
y_t = r_t + \gamma (1-d_t) \max_a Q_{\text{target}}(s_{t+1}, a),
$$

并最小化

$$
\mathcal{L}_{\text{DQN}} = \mathbb{E}[(Q(s_t,a_t)-y_t)^2].
$$

下面的函数创建一个很小的合成 batch，并调用仓库中真实的 `DQNAgent.update()`。它只验证数据契约和更新路径，不代表 Atari 学习效果。

In [ ]:
run_dqn_update_demo()

## 4. PPO：GAE 与 clipped objective

优势估计使用

$$
\delta_t = r_t + \gamma(1-d_t)V(s_{t+1}) - V(s_t),
$$

$$
A_t = \delta_t + \gamma\lambda(1-d_t)A_{t+1}.
$$

PPO 再通过概率比 $r_t(\theta)$ 的 clipped objective 限制单次更新幅度：

$$
\mathcal{L}_{\text{clip}} = \mathbb{E}[\min(r_t A_t, \operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t)].
$$

下一单元格直接调用 `algorithms.ppo.generalized_advantage_estimate` 的固定输入示例。

In [ ]:
run_gae_demo()

## 5. EWC：稳定性正则项，而不是冲突求解器

任务结束后，PPO 使用策略负对数似然的逐样本平方梯度估计对角经验 Fisher；DQN 没有对应的策略似然，因此使用逐样本 TD-MSE 平方梯度作为重要性近似：

$$
F_i \approx \frac{1}{N}\sum_{n=1}^{N}\left(\frac{\partial \ell_n}{\partial \theta_i}\right)^2.
$$

后续任务增加二次惩罚：

$$
\mathcal{L}_{\text{EWC}} = \frac{\lambda}{2}\sum_i I_i(\theta_i-\theta_i^*)^2.
$$

其中 PPO 的 $I_i=F_i$ 只覆盖策略敏感的 backbone 和 actor，不覆盖 critic；DQN 的 $I_i$ 是 TD-gradient surrogate。下面只演示数学机制：权重位于已巩固参考点时 penalty 为零，移动重要权重后 penalty 增大。这个结果不能证明 EWC 改善了旧任务，也不能证明它解决了新旧任务的梯度冲突。

In [ ]:
run_ewc_penalty_demo()

## 6. 正式实验使用 Python runner

长时间实验应在终端执行，而不是嵌入 Notebook：

```bash
python scripts/run_experiments.py train single
python scripts/run_experiments.py train continual --parallel
python scripts/run_experiments.py evaluate continual
```

课堂上可以用 `--dry-run` 预览完整任务矩阵。下一单元格调用的仍然是同一个 Python runner，但不会启动训练。

In [ ]:
from scripts.run_experiments import main as run_experiment_matrix

run_experiment_matrix(("train", "continual", "--device", "cpu", "--dry-run"))

## 7. 持续学习的正确评估对象

训练阶段 $i$ 完成后，在每个已见任务 $j$ 上评估，得到阶段×任务矩阵 $R[i,j]$。至少分别报告：

- **Plasticity**：任务 $j$ 首次训练后的 $R[j,j]$；
- **Retention**：该任务历史最佳分数与最终分数之差；
- **Trade-off**：同一方法的 plasticity 与 retention，而不是只看最终平均分。

不同 Atari 游戏的原始奖励尺度不同，不能直接跨游戏平均。正式训练会生成 `continual_evaluation.json`，下面的薄入口只在结果存在时调用公共绘图函数。

In [ ]:
from pathlib import Path
from scripts.visualize_results import plot_continual_results

if list(Path("outputs").glob("**/continual_evaluation.json")):
    plot_continual_results("outputs")
else:
    print("No continual_evaluation.json found; run a continual experiment first.")

## 8. 结论与下一步

Notebook 中的短示例只验证实现路径。算法结论必须来自固定 seeds、固定评估预算和保存完整 $R[i,j]$ 的正式实验。

当前优先比较 EWC 与小型 episodic replay：EWC 主要限制旧任务重要参数的漂移，而 replay 更直接地向新任务更新提供旧任务约束。只有在这两个基线协议稳定后，再考虑梯度投影、蒸馏或更复杂的 TorchRL 组件。